In [2]:
import pandas as pd

# Load datasets
departments = pd.read_csv("departments_clean.csv")
doctors = pd.read_csv("doctors_preprocessed.csv")
geocodes = pd.read_csv("facility_geocodes.csv")

print("DEPARTMENTS")
print(departments.columns.tolist())
print(departments.shape)

print("\nDOCTORS")
print(doctors.columns.tolist())
print(doctors.shape)

print("\nFACILITY GEOCODES")
print(geocodes.columns.tolist())
print(geocodes.shape)

DEPARTMENTS
['department_id', 'department_name']
(20, 2)

DOCTORS
['Doctor_ID', 'Doctor_Name', 'Department_ID', 'Specialization', 'Experience']
(500, 5)

FACILITY GEOCODES
['department_id', 'department_name', 'latitude', 'longitude']
(20, 4)


In [3]:
# Check department IDs in all three datasets

print("Departments IDs:")
print(sorted(departments["department_id"].unique()))

print("\nGeocode IDs:")
print(sorted(geocodes["department_id"].unique()))

print("\nDoctor Department IDs:")
print(sorted(doctors["Department_ID"].unique()))

Departments IDs:
['D001', 'D002', 'D003', 'D004', 'D005', 'D006', 'D007', 'D008', 'D009', 'D010', 'D011', 'D012', 'D013', 'D014', 'D015', 'D016', 'D017', 'D018', 'D019', 'D020']

Geocode IDs:
['D001', 'D002', 'D003', 'D004', 'D005', 'D006', 'D007', 'D008', 'D009', 'D010', 'D011', 'D012', 'D013', 'D014', 'D015', 'D016', 'D017', 'D018', 'D019', 'D020']

Doctor Department IDs:
['D001', 'D002', 'D003', 'D004', 'D005', 'D006', 'D007', 'D008', 'D009', 'D010', 'D011', 'D012', 'D013', 'D014', 'D015', 'D016', 'D017', 'D018', 'D019', 'D020']


In [4]:
# Check whether every department has a geocode
department_ids = set(departments["department_id"])
geocode_ids = set(geocodes["department_id"])

missing_geocodes = department_ids - geocode_ids
extra_geocodes = geocode_ids - department_ids

print("Departments without geocodes:", missing_geocodes)
print("Geocodes without matching departments:", extra_geocodes)

Departments without geocodes: set()
Geocodes without matching departments: set()


In [5]:
# Check whether every doctor belongs to a valid department

doctor_department_ids = set(doctors["Department_ID"])

invalid_doctor_departments = doctor_department_ids - department_ids

print("Doctor department IDs not found in departments:")
print(invalid_doctor_departments)

Doctor department IDs not found in departments:
set()


In [6]:
department_geo = pd.merge(
    departments,
    geocodes,
    on="department_id",
    how="inner"
)

print("Department + Geocode dataset:")
print(department_geo.head())

print("\nShape:", department_geo.shape)

Department + Geocode dataset:
  department_id department_name_x department_name_y  latitude  longitude
0          D001        Cardiology        Cardiology   17.4239    78.4487
1          D002         Neurology         Neurology   17.3997    78.4760
2          D003       Orthopedics       Orthopedics   17.3715    78.4870
3          D004        Pediatrics        Pediatrics   17.4401    78.4980
4          D005          Oncology          Oncology   17.3900    78.5200

Shape: (20, 5)


In [7]:
# Keep the department name from departments_clean.csv
department_geo = department_geo.rename(
    columns={"department_name_x": "department_name"}
)

# Remove the duplicate department name from geocodes
department_geo = department_geo.drop(
    columns=["department_name_y"]
)

print("Clean Department + Geocode dataset:")
display(department_geo.head())

print("\nShape:", department_geo.shape)
print("\nColumns:", department_geo.columns.tolist())

Clean Department + Geocode dataset:


,department_id,department_name,latitude,longitude
0,D001,Cardiology,17.4239,78.4487
1,D002,Neurology,17.3997,78.4760
2,D003,Orthopedics,17.3715,78.4870
3,D004,Pediatrics,17.4401,78.4980
4,D005,Oncology,17.3900,78.5200



Shape: (20, 4)

Columns: ['department_id', 'department_name', 'latitude', 'longitude']


In [8]:
# Rename the doctor department column for joining
doctors_analysis = doctors.rename(
    columns={"Department_ID": "department_id"}
)

# Join doctors with department location data
service_data = pd.merge(
    department_geo,
    doctors_analysis,
    on="department_id",
    how="left"
)

print("Service Coverage Dataset:")
display(service_data.head())

print("\nShape:", service_data.shape)

Service Coverage Dataset:


,department_id,department_name,latitude,longitude,Doctor_ID,Doctor_Name,Specialization,Experience
0,D001,Cardiology,17.4239,78.4487,DR00011,Doctor_11,Cardiology,22
1,D001,Cardiology,17.4239,78.4487,DR00031,Doctor_31,Cardiology,9
2,D001,Cardiology,17.4239,78.4487,DR00034,Doctor_34,Cardiology,28
3,D001,Cardiology,17.4239,78.4487,DR00070,Doctor_70,Cardiology,13
4,D001,Cardiology,17.4239,78.4487,DR00075,Doctor_75,Cardiology,15



Shape: (500, 8)


In [9]:
# Calculate doctor/service availability by department

department_service = (
    service_data
    .groupby(
        ["department_id", "department_name", "latitude", "longitude"],
        as_index=False
    )
    .agg(
        Doctor_Count=("Doctor_ID", "nunique"),
        Average_Experience=("Experience", "mean"),
        Maximum_Experience=("Experience", "max")
    )
)

# Round experience values
department_service["Average_Experience"] = (
    department_service["Average_Experience"].round(1)
)

print("Department-level Service Coverage Analysis:")
display(department_service)

Department-level Service Coverage Analysis:


,department_id,department_name,latitude,longitude,Doctor_Count,Average_Experience,Maximum_Experience
0,D001,Cardiology,17.4239,78.4487,23,17.6,32
1,D002,Neurology,17.3997,78.4760,20,19.2,34
2,D003,Orthopedics,17.3715,78.4870,30,15.4,35
3,D004,Pediatrics,17.4401,78.4980,27,18.3,35
4,D005,Oncology,17.3900,78.5200,33,19.8,35
5,D006,ENT,17.4550,78.4700,21,16.8,32
6,D007,Dermatology,17.3620,78.4500,20,16.7,35
7,D008,General Surgery,17.4100,78.5300,18,19.0,35
8,D009,Urology,17.3500,78.5000,17,20.8,35
9,D010,Nephrology,17.4300,78.5200,22,18.9,34


In [10]:
# Doctor availability by department

department_service = (
    service_data
    .groupby(
        ["department_id", "department_name", "latitude", "longitude"],
        as_index=False
    )
    .agg(
        Doctor_Count=("Doctor_ID", "nunique"),
        Average_Experience=("Experience", "mean")
    )
)

# Round average experience
department_service["Average_Experience"] = (
    department_service["Average_Experience"].round(1)
)

# Calculate the average number of doctors across departments
average_doctors = department_service["Doctor_Count"].mean()

print("Average doctors per department:", round(average_doctors, 1))

display(department_service)

Average doctors per department: 25.0


,department_id,department_name,latitude,longitude,Doctor_Count,Average_Experience
0,D001,Cardiology,17.4239,78.4487,23,17.6
1,D002,Neurology,17.3997,78.4760,20,19.2
2,D003,Orthopedics,17.3715,78.4870,30,15.4
3,D004,Pediatrics,17.4401,78.4980,27,18.3
4,D005,Oncology,17.3900,78.5200,33,19.8
5,D006,ENT,17.4550,78.4700,21,16.8
6,D007,Dermatology,17.3620,78.4500,20,16.7
7,D008,General Surgery,17.4100,78.5300,18,19.0
8,D009,Urology,17.3500,78.5000,17,20.8
9,D010,Nephrology,17.4300,78.5200,22,18.9


In [11]:
# Classify departments based on doctor availability

department_service["Service_Availability"] = department_service[
    "Doctor_Count"
].apply(
    lambda x: "High" if x >= average_doctors else "Low"
)

display(department_service)

,department_id,department_name,latitude,longitude,Doctor_Count,Average_Experience,Service_Availability
0,D001,Cardiology,17.4239,78.4487,23,17.6,Low
1,D002,Neurology,17.3997,78.4760,20,19.2,Low
2,D003,Orthopedics,17.3715,78.4870,30,15.4,High
3,D004,Pediatrics,17.4401,78.4980,27,18.3,High
4,D005,Oncology,17.3900,78.5200,33,19.8,High
5,D006,ENT,17.4550,78.4700,21,16.8,Low
6,D007,Dermatology,17.3620,78.4500,20,16.7,Low
7,D008,General Surgery,17.4100,78.5300,18,19.0,Low
8,D009,Urology,17.3500,78.5000,17,20.8,Low
9,D010,Nephrology,17.4300,78.5200,22,18.9,Low


In [12]:
# Benchmark service radius for each department

SERVICE_RADIUS_KM = 5

department_service["Service_Radius_KM"] = SERVICE_RADIUS_KM

# Calculate approximate circular catchment area
department_service["Catchment_Area_Sq_KM"] = (
    3.14159 * department_service["Service_Radius_KM"] ** 2
)

display(department_service)

,department_id,department_name,latitude,longitude,Doctor_Count,Average_Experience,Service_Availability,Service_Radius_KM,Catchment_Area_Sq_KM
0,D001,Cardiology,17.4239,78.4487,23,17.6,Low,5,78.53975
1,D002,Neurology,17.3997,78.4760,20,19.2,Low,5,78.53975
2,D003,Orthopedics,17.3715,78.4870,30,15.4,High,5,78.53975
3,D004,Pediatrics,17.4401,78.4980,27,18.3,High,5,78.53975
4,D005,Oncology,17.3900,78.5200,33,19.8,High,5,78.53975
5,D006,ENT,17.4550,78.4700,21,16.8,Low,5,78.53975
6,D007,Dermatology,17.3620,78.4500,20,16.7,Low,5,78.53975
7,D008,General Surgery,17.4100,78.5300,18,19.0,Low,5,78.53975
8,D009,Urology,17.3500,78.5000,17,20.8,Low,5,78.53975
9,D010,Nephrology,17.4300,78.5200,22,18.9,Low,5,78.53975


In [13]:
# Identify better-served and under-served departments
# based on doctor availability

average_doctors = department_service["Doctor_Count"].mean()

department_service["Service_Coverage_Status"] = department_service[
    "Doctor_Count"
].apply(
    lambda x: "Better Served" if x >= average_doctors else "Under-Served"
)

print("Average number of doctors per department:", round(average_doctors, 2))

display(
    department_service[
        [
            "department_id",
            "department_name",
            "Doctor_Count",
            "Service_Radius_KM",
            "Catchment_Area_Sq_KM",
            "Service_Coverage_Status"
        ]
    ]
)

Average number of doctors per department: 25.0


,department_id,department_name,Doctor_Count,Service_Radius_KM,Catchment_Area_Sq_KM,Service_Coverage_Status
0,D001,Cardiology,23,5,78.53975,Under-Served
1,D002,Neurology,20,5,78.53975,Under-Served
2,D003,Orthopedics,30,5,78.53975,Better Served
3,D004,Pediatrics,27,5,78.53975,Better Served
4,D005,Oncology,33,5,78.53975,Better Served
5,D006,ENT,21,5,78.53975,Under-Served
6,D007,Dermatology,20,5,78.53975,Under-Served
7,D008,General Surgery,18,5,78.53975,Under-Served
8,D009,Urology,17,5,78.53975,Under-Served
9,D010,Nephrology,22,5,78.53975,Under-Served


In [14]:
# Doctor Density per catchment area
department_service["Doctors_Per_100_Sq_KM"] = (
    department_service["Doctor_Count"]
    / department_service["Catchment_Area_Sq_KM"]
) * 100

display(
    department_service[
        [
            "department_id",
            "department_name",
            "Doctor_Count",
            "Service_Radius_KM",
            "Catchment_Area_Sq_KM",
            "Doctors_Per_100_Sq_KM"
        ]
    ].sort_values(
        by="Doctors_Per_100_Sq_KM",
        ascending=False
    )
)

,department_id,department_name,Doctor_Count,Service_Radius_KM,Catchment_Area_Sq_KM,Doctors_Per_100_Sq_KM
14,D015,Emergency,34,5,78.53975,43.290181
4,D005,Oncology,33,5,78.53975,42.016940
18,D019,Endocrinology,32,5,78.53975,40.743700
2,D003,Orthopedics,30,5,78.53975,38.197219
12,D013,Pulmonology,29,5,78.53975,36.923978
3,D004,Pediatrics,27,5,78.53975,34.377497
17,D018,Ophthalmology,27,5,78.53975,34.377497
15,D016,Radiology,27,5,78.53975,34.377497
13,D014,Gastroenterology,27,5,78.53975,34.377497
11,D012,Psychiatry,27,5,78.53975,34.377497


In [16]:

import numpy as np
# Calculate distance between two geographic coordinates
def calculate_distance_km(lat1, lon1, lat2, lon2):
    R = 6371  # Earth's radius in km

    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)

    dlat = lat2 - lat1
    dlon = np.radians(lon2) - np.radians(lon1)

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c


# Create all department pairs
department_pairs = department_service.merge(
    department_service,
    on=None,
    how="cross",
    suffixes=("_1", "_2")
)

# Remove comparisons of a department with itself
department_pairs = department_pairs[
    department_pairs["department_id_1"] != department_pairs["department_id_2"]
].copy()

# Keep each pair only once
department_pairs["pair_key"] = department_pairs.apply(
    lambda row: tuple(
        sorted([row["department_id_1"], row["department_id_2"]])
    ),
    axis=1
)

department_pairs = department_pairs.drop_duplicates(
    subset="pair_key"
)

# Calculate geographic distance
department_pairs["Distance_KM"] = calculate_distance_km(
    department_pairs["latitude_1"],
    department_pairs["longitude_1"],
    department_pairs["latitude_2"],
    department_pairs["longitude_2"]
)

# Select useful columns
department_distances = department_pairs[
    [
        "department_id_1",
        "department_name_1",
        "department_id_2",
        "department_name_2",
        "Distance_KM"
    ]
].sort_values(
    by="Distance_KM"
).reset_index(drop=True)

display(department_distances.head(20))

,department_id_1,department_name_1,department_id_2,department_name_2,Distance_KM
0,D005,Oncology,D018,Ophthalmology,1.685974
1,D007,Dermatology,D011,Gynecology,2.070650
2,D001,Cardiology,D013,Pulmonology,2.205321
3,D002,Neurology,D013,Pulmonology,2.304830
4,D006,ENT,D020,Dental,2.305087
5,D005,Oncology,D016,Radiology,2.305569
6,D008,General Surgery,D010,Nephrology,2.464009
7,D005,Oncology,D008,General Surgery,2.464059
8,D004,Pediatrics,D010,Nephrology,2.590052
9,D003,Orthopedics,D009,Urology,2.760240


In [18]:
# Classify potential service-area overlap

def classify_overlap(distance):
    if distance <= 5:
        return "High Overlap Potential"
    elif distance <= 10:
        return "Moderate Overlap Potential"
    else:
        return "Low / No Overlap Potential"


department_distances["Overlap_Level"] = (
    department_distances["Distance_KM"]
    .apply(classify_overlap)
)

print("Potential Service-Area Overlap:")
display(
    department_distances[
        [
            "department_id_1",
            "department_name_1",
            "department_id_2",
            "department_name_2",
            "Distance_KM",
            "Overlap_Level"
        ]
    ]
)

Potential Service-Area Overlap:


,department_id_1,department_name_1,department_id_2,department_name_2,Distance_KM,Overlap_Level
0,D005,Oncology,D018,Ophthalmology,1.685974,High Overlap Potential
1,D007,Dermatology,D011,Gynecology,2.070650,High Overlap Potential
2,D001,Cardiology,D013,Pulmonology,2.205321,High Overlap Potential
3,D002,Neurology,D013,Pulmonology,2.304830,High Overlap Potential
4,D006,ENT,D020,Dental,2.305087,High Overlap Potential
...,...,...,...,...,...,...
185,D006,ENT,D019,Endocrinology,14.309977,Low / No Overlap Potential
186,D012,Psychiatry,D014,Gastroenterology,14.696718,Low / No Overlap Potential
187,D014,Gastroenterology,D017,ICU,14.788164,Low / No Overlap Potential
188,D014,Gastroenterology,D020,Dental,15.160571,Low / No Overlap Potential


In [19]:
print("OVERLAP SUMMARY")
print("--------------------------------")

print(
    department_distances["Overlap_Level"]
    .value_counts()
)

print("\nTotal department pairs:", len(department_distances))

OVERLAP SUMMARY
--------------------------------
Overlap_Level
Moderate Overlap Potential    95
Low / No Overlap Potential    49
High Overlap Potential        46
Name: count, dtype: int64

Total department pairs: 190


In [25]:
# Final prepared dataset for Member 3

final_service_coverage = department_service[
    [
        "department_id",
        "department_name",
        "latitude",
        "longitude",
        "Doctor_Count",
        "Average_Experience",
        "Service_Availability",
        "Service_Radius_KM",
        "Catchment_Area_Sq_KM",
        "Service_Coverage_Status"
    ]
].copy()

# Save prepared dataset
final_service_coverage.to_csv(
    "service_coverage_prepared.csv",
    index=False
)

print("✅ service_coverage_prepared.csv created successfully")
print("Shape:", final_service_coverage.shape)

display(final_service_coverage)

✅ service_coverage_prepared.csv created successfully
Shape: (20, 10)


,department_id,department_name,latitude,longitude,Doctor_Count,Average_Experience,Service_Availability,Service_Radius_KM,Catchment_Area_Sq_KM,Service_Coverage_Status
0,D001,Cardiology,17.4239,78.4487,23,17.6,Low,5,78.53975,Under-Served
1,D002,Neurology,17.3997,78.4760,20,19.2,Low,5,78.53975,Under-Served
2,D003,Orthopedics,17.3715,78.4870,30,15.4,High,5,78.53975,Better Served
3,D004,Pediatrics,17.4401,78.4980,27,18.3,High,5,78.53975,Better Served
4,D005,Oncology,17.3900,78.5200,33,19.8,High,5,78.53975,Better Served
5,D006,ENT,17.4550,78.4700,21,16.8,Low,5,78.53975,Under-Served
6,D007,Dermatology,17.3620,78.4500,20,16.7,Low,5,78.53975,Under-Served
7,D008,General Surgery,17.4100,78.5300,18,19.0,Low,5,78.53975,Under-Served
8,D009,Urology,17.3500,78.5000,17,20.8,Low,5,78.53975,Under-Served
9,D010,Nephrology,17.4300,78.5200,22,18.9,Low,5,78.53975,Under-Served


In [26]:
# Key findings for Member 3

average_doctors = final_service_coverage["Doctor_Count"].mean()

most_doctors = final_service_coverage.loc[
    final_service_coverage["Doctor_Count"].idxmax()
]

least_doctors = final_service_coverage.loc[
    final_service_coverage["Doctor_Count"].idxmin()
]

better_served_count = (
    final_service_coverage["Service_Coverage_Status"]
    .eq("Better Served")
    .sum()
)

under_served_count = (
    final_service_coverage["Service_Coverage_Status"]
    .eq("Under-Served")
    .sum()
)

print("MEMBER 3 – HEALTHCARE SERVICE COVERAGE FINDINGS")
print("=" * 55)

print(f"Average doctors per department: {average_doctors:.0f}")
print(f"Benchmark service radius: {final_service_coverage['Service_Radius_KM'].iloc[0]} km")
print(
    f"Benchmark catchment area per department: "
    f"{final_service_coverage['Catchment_Area_Sq_KM'].iloc[0]:.2f} sq km"
)

print(
    f"\nDepartment with highest doctor availability: "
    f"{most_doctors['department_name']} "
    f"({most_doctors['Doctor_Count']} doctors)"
)

print(
    f"Department with lowest doctor availability: "
    f"{least_doctors['department_name']} "
    f"({least_doctors['Doctor_Count']} doctors)"
)

print(f"\nBetter Served departments: {better_served_count}")
print(f"Under-Served departments: {under_served_count}")

MEMBER 3 – HEALTHCARE SERVICE COVERAGE FINDINGS
Average doctors per department: 25
Benchmark service radius: 5 km
Benchmark catchment area per department: 78.54 sq km

Department with highest doctor availability: Emergency (34 doctors)
Department with lowest doctor availability: Urology (17 doctors)

Better Served departments: 11
Under-Served departments: 9
